In [39]:
import os, pathlib, textwrap, glob

from langchain_community.document_loaders import PyPDFLoader, TextLoader, UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import (
    OpenAIEmbeddings,
    HuggingFaceBgeEmbeddings,
    SentenceTransformerEmbeddings,
)
from langchain_community.llms import Ollama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaLLM

In [40]:
pdf_paths = glob.glob("data/*.*")
raw_docs = []

for path in pdf_paths:
    load_doc = PyPDFLoader(path).load()
    raw_docs.extend(load_doc)

raw_docs


Ignoring wrong pointing object 81 0 (offset 0)
Ignoring wrong pointing object 76 0 (offset 0)
Ignoring wrong pointing object 80 0 (offset 0)


[Document(metadata={'producer': 'Skia/PDF m138 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Return_and_exchange_policy', 'source': 'data/Everstorm_Return_and_exchange_policy.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Everstorm  Outfitters    RETURN  &  EXCHANGE  POLICY    Document  ROX-2025-05   Easy-Fit  Promise    If  your  gear  doesn’t  fit  or  just  isn’t  your  vibe,  send  it  back  within  **30  days**  of  delivery  for  a  refund  or  free  size  exchange.   Eligibility  Checklist    ●  Unworn,  unwashed,  no  odors,  tags  attached    ●  Original  shoe  box  (footwear)  placed  inside  outer  carton    ●  Electronics  (power-banks,  headlamps)  unopened  unless  faulty   How  to  Start    ●  Visit  everstorm.example/returns  →  enter  order  #  and  email.    ●  Select  “Refund”  or  “Exchange.”    ●  Print  prepaid  label;  pack  securely.  Multiple  items  can  share  one  box.   Instant  Exchange  Hold    We  place  a

In [41]:
from langchain_community.document_loaders import WebBaseLoader


urls = ["https://poe2db.tw/us/", "https://reference.langchain.com" ]


try:
    loader = WebBaseLoader(urls)
    url_docs = loader.load()
    print(url_docs)
except:
    print("Failed")

[Document(metadata={'source': 'https://poe2db.tw/us/', 'title': 'Home - PoE2DB, Path of Exile Wiki us', 'description': 'path of exile is simply a tool to test your path of building creation', 'language': 'us'}, page_content="\n\n\n\n\n\nHome - PoE2DB, Path of Exile Wiki us\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n        Update cookie preferences\n      \n\n\n\n\n\n\n\n\n\n\nPoE2DB\n\n\n\n\n\n\nItem \n\nItem\n\n\n\nGem\nSkill Gems\nSupport Gems\nSpirit Gems\nLineage Supports\n\n\n\nModifiers \n\nModifiers\nDesecrated Modifiers\nKeywords\nCrafting\n\n\n\n\nQuest \n\nQuest\nAscendancy Classes\nPassive Skill Tree\nAct\n\nWaystones\n\n\n\n\nEconomy\nPatreon\n\n\n\nPoEDB\n\n\n  \n\nTW 繁體中文\nCN 简体中文\nUS English\nKR 한국어\nJP Japanese\nRU Русский\nPO Português\nTH ภาษาไทย\nFR Français\nDE Deutsch\nES Spanish\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nReturn of the Ancients 0.5 Patch NotesStarts in \nThe Last of the Druids 0.4 Running for

In [42]:
chunks = []
textSplitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = textSplitter.split_documents(raw_docs)

In [43]:
embedding_vector = []

embedding_model = HuggingFaceEmbeddings(model="sentence-transformers/all-mpnet-base-v2")

In [64]:
vectorb = FAISS.from_documents(chunks, embedding_model)
retriever = vectorb.as_retriever(search_kwargs={"k" : 8})
vectorb.save_local("faiss_index")

vectorb.index.ntotal

42

In [49]:
llm = OllamaLLM(model="llama3.2", temperature=0.5)

In [46]:
SYSTEM_TEMPLATE = """
You are a **Customer Support Charbot**. Use only the information in CONTEXT to answer.
If the answer is not in CONTEXT, respond with "Im not sure from the docs."

Rules:
1. Use only the provided <context> to anser.
2. If the answer is not in the context, say "I dont know based on the retrieved documents."
3. Be concise and accurate. Prefer quotnig key phrases from the context.
4. When possible, cite sources as [source : source] using the metadata.response

CONTEXT:
{context}

USER:
{question}
"""

In [51]:
prompt = PromptTemplate(input_variables = ["context", "question"], template = SYSTEM_TEMPLATE)
formatted = prompt.format(context="no refund policy", question="What is the refund policy ?")

In [54]:
chain = ConversationalRetrievalChain.from_llm(llm, retriever, combine_docs_chain_kwargs={"prompt" : prompt}, return_source_documents=True)

chain

ConversationalRetrievalChain(verbose=False, combine_docs_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are a **Customer Support Charbot**. Use only the information in CONTEXT to answer.\nIf the answer is not in CONTEXT, respond with "Im not sure from the docs."\n\nRules:\n1. Use only the provided <context> to anser.\n2. If the answer is not in the context, say "I dont know based on the retrieved documents."\n3. Be concise and accurate. Prefer quotnig key phrases from the context.\n4. When possible, cite sources as [source : source] using the metadata.response\n\nCONTEXT:\n{context}\n\nUSER:\n{question}\n'), llm=OllamaLLM(model='llama3.2', temperature=0.5), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_

In [62]:
test_questions = [
    "If im not happy with my purchase, what is your refund policy ?",
    "How do I start my return ?",
    "Will i get the full price ?"
]

chat_history = []

for q in test_questions:
    result = chain.invoke({"question": q, "chat_history": chat_history})
    print(result["answer"])
    chat_history.append((q, result["answer"]))


"and custom-embroidered items: no return unless defective." [source : Everstorm Outfitters RETURN & EXCHANGE POLICY]

Refunds are issued the same day your return is scanned.
To initiate a return, follow these steps: Select "Refund" or "Exchange." Print prepaid label; pack securely. Multiple items can share one box.
According to the context, "and custom-embroidered items: no return unless defective." This implies that if an item is not defective (e.g., it's worn out), you won't receive the full purchase price for your returned item. [source : Everstorm Outfitters RETURN & EXCHANGE POLICY]


In [63]:
chat_history

[('If im not happy with my purchase, what is your refund policy ?',
  '"and custom-embroidered items: no return unless defective." [source : Everstorm Outfitters RETURN & EXCHANGE POLICY]\n\nRefunds are issued the same day your return is scanned.'),
 ('How do I start my return ?',
  'To initiate a return, follow these steps: Select "Refund" or "Exchange." Print prepaid label; pack securely. Multiple items can share one box.'),
 ('Will i get the full price ?',
  'According to the context, "and custom-embroidered items: no return unless defective." This implies that if an item is not defective (e.g., it\'s worn out), you won\'t receive the full purchase price for your returned item. [source : Everstorm Outfitters RETURN & EXCHANGE POLICY]')]